## 1. Import Libraries




In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sqlalchemy import create_engine
import glob
import os
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
import statsmodels.api as sm

## 2. Load Daily Records

Load the station-day pollutant and weather summaries from MySQL `air_quality.aqi_daily`. All exceedance and station comparisons below use these daily records.


In [ ]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")

    df = pd.read_sql_query("SELECT * FROM aqi_daily", engine)
    combined_df = df.copy()
    print(f"Loaded {len(df):,} daily station rows from aqi_daily.")
except Exception as error:
    raise RuntimeError(
        "Could not connect to MySQL or load the aqi_daily table. "
        "Check that MySQL is running and air_quality.aqi_daily exists."
    ) from error

## 3. Descriptive Statistics

Calculate mean, median, standard deviation, variance, min, max, Q1, Q3, and IQR for each pollutant. Describe the overall numerical characteristics of the air-quality data.

In [ ]:
pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]

descriptive_stats = df[pollutants].agg([
    "count",
    "mean",
    "median",
    "std",
    "var",
    "min",
    "max"
])

q1 = df[pollutants].quantile(0.25)
q3 = df[pollutants].quantile(0.75)

descriptive_stats.loc["Q1"] = q1
descriptive_stats.loc["Q3"] = q3
descriptive_stats.loc["IQR"] = q3 - q1

descriptive_stats = descriptive_stats.T

descriptive_stats

In [ ]:
confidence_results = []

for pollutant in pollutants:
    values = df[pollutant].dropna()
    n = len(values)
    mean = values.mean()
    standard_error = values.std() / np.sqrt(n)
    ci_low, ci_high = sm.stats.DescrStatsW(values).tconfint_mean(
        alpha=0.05
    )

    confidence_results.append({
        "Pollutant": pollutant,
        "Mean": mean,
        "95% CI Lower": ci_low,
        "95% CI Upper": ci_high
    })

confidence_df = pd.DataFrame(confidence_results)

confidence_df

## 4. Station-Level Analysis 


In [ ]:
station_stats = df.groupby("station")[pollutants].agg([
    "mean",
    "median",
    "std"
])

station_stats

In [ ]:
station_summary = []

for station, group in df.groupby("station"):
    for pollutant in pollutants:
        values = group[pollutant].dropna()
        station_summary.append({
            "Station": station,
            "Pollutant": pollutant,
            "Mean": values.mean(),
            "Median": values.median(),
            "Standard Deviation": values.std()
        })

station_summary_df = pd.DataFrame(station_summary)

station_summary_df

In [ ]:
highest_mean_results = []

for pollutant in pollutants:
    station_means = df.groupby("station")[pollutant].mean()
    highest_station = station_means.idxmax()
    highest_mean = station_means.max()

    highest_mean_results.append({
        "Pollutant": pollutant,
        "Station with Highest Mean": highest_station,
        "Mean Concentration": highest_mean
    })

highest_mean_df = pd.DataFrame(highest_mean_results)

highest_mean_df

In [ ]:
station_confidence = []

for station, group in df.groupby("station"):
    for pollutant in pollutants:
        values = group[pollutant].dropna()

        if len(values) > 1:
            ci_low, ci_high = sm.stats.DescrStatsW(
                values
            ).tconfint_mean(alpha=0.05)

            station_confidence.append({
                "Station": station,
                "Pollutant": pollutant,
                "Mean": values.mean(),
                "95% CI Lower": ci_low,
                "95% CI Upper": ci_high
            })

station_confidence_df = pd.DataFrame(station_confidence)

pd.set_option("display.max_rows", None)

## WHO Guideline Exceedance Analysis

Compare daily pollutant summaries with the WHO 2021 guideline values. `O3` in `aqi_daily` represents the daily maximum 8-hour mean; `CO` is recorded in mg/m³. Exceedance percentages are calculated across valid station-days for each station.


In [ ]:
who_guidelines = {
    "PM2.5": 15.0,  # 24-hour mean, ug/m3
    "PM10": 45.0,   # 24-hour mean, ug/m3
    "SO2": 40.0,    # 24-hour mean, ug/m3
    "NO2": 25.0,    # 24-hour mean, ug/m3
    "O3": 100.0,    # daily maximum 8-hour mean, ug/m3
    "CO": 4.0,      # 24-hour mean, mg/m3
}
metric_names = {
    "PM2.5": "24-hour mean",
    "PM10": "24-hour mean",
    "SO2": "24-hour mean",
    "NO2": "24-hour mean",
    "O3": "daily maximum 8-hour mean",
    "CO": "24-hour mean",
}

required = ["station", *who_guidelines]
missing = [column for column in required if column not in df.columns]
if missing:
    raise KeyError(f"aqi_daily is missing WHO analysis columns: {missing}")

exceedance_rows = []
for station, station_data in df.groupby("station", dropna=True):
    for pollutant, guideline in who_guidelines.items():
        values = pd.to_numeric(station_data[pollutant], errors="coerce").dropna()
        exceedances = int(values.gt(guideline).sum())
        exceedance_rows.append({
            "station": station,
            "pollutant": pollutant,
            "metric": metric_names[pollutant],
            "exceedances": exceedances,
            "valid_days": len(values),
            "exceedance_percentage": 100 * exceedances / len(values) if len(values) else np.nan,
        })

who_exceedance_df = pd.DataFrame(exceedance_rows)
who_exceedance_df.head()

In [ ]:
who_exceedance_summary = (
    who_exceedance_df.groupby("pollutant")
    .agg(
        metric=("metric", "first"),
        mean_exceedance_percentage=("exceedance_percentage", "mean"),
        max_exceedance_percentage=("exceedance_percentage", "max"),
        stations_exceeding=("exceedances", lambda x: (x > 0).sum()),
        total_stations=("station", "nunique"),
    )
    .reset_index()
)
who_exceedance_summary[["mean_exceedance_percentage", "max_exceedance_percentage"]] = (
    who_exceedance_summary[["mean_exceedance_percentage", "max_exceedance_percentage"]].round(2)
)
 
who_exceedance_summary


### Interpretation

The table above reports the share of valid station-days whose daily summary exceeds each guideline value. These are comparisons of the stored daily summaries; they do not reconstruct hourly exposures or calculate a fresh rolling 8-hour window. Interpret the results as screening statistics, not as a health-risk assessment.


## 5. Station Differences (ANOVA)

One-way ANOVA compares mean daily pollutant summaries among stations, followed by Bonferroni correction across pollutants and Tukey HSD station-pair comparisons. Since station records on the same date share regional conditions, the independence assumption may not hold; treat p-values as exploratory evidence of station differences.


In [ ]:
from scipy.stats import f_oneway, levene, normaltest
from statsmodels.stats.multicomp import pairwise_tukeyhsd

pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]

for pollutant in pollutants:
    df[pollutant] = pd.to_numeric(df[pollutant], errors="coerce")

anova_results = []

for pollutant in pollutants:
    station_groups = [
        group[pollutant].dropna().to_numpy()
        for _, group in df.groupby("station", sort=True)
    ]
    f_statistic, p_value = f_oneway(*station_groups)
    anova_results.append({
        "pollutant": pollutant,
        "stations": len(station_groups),
        "observations": sum(len(values) for values in station_groups),
        "F-statistic": f_statistic,
        "p-value": p_value,
    })

anova_results = pd.DataFrame(anova_results)
anova_results["p-value (Bonferroni)"] = multipletests(
    anova_results["p-value"], method="bonferroni"
)[1]
anova_results["significant (alpha=0.05)"] = (
    anova_results["p-value (Bonferroni)"] < 0.05
)

anova_results.round({
    "F-statistic": 3,
    "p-value": 6,
    "p-value (Bonferroni)": 6,
})

## 6. ANOVA Interpretation

Use the fresh ANOVA table above to assess evidence of station differences for each pollutant. The test compares daily station summaries and its p-values are exploratory because same-date observations across stations are not fully independent.


## 7. Post-Hoc Explanation

A significant ANOVA result shows that at least one station mean differs, but it does not identify the station pairs responsible for the differences. Tukey's HSD compares every pair of stations for each pollutant while controlling the family-wise error rate at 0.05.

The same analysis checks two ANOVA assumptions: Levene's test evaluates whether station variances are similar, and the residual normality test evaluates whether residuals are approximately normal. The full pairwise results are retained in `significant_pairwise_results`, while the next cell displays only a compact summary.

## 8. Compact Post-Hoc Results

The output shows the number of significant station pairs for each pollutant, the three largest significant mean differences per pollutant, and the assumption-test results.

In [ ]:
significant_pairwise_results = []
assumption_results = []

for pollutant in pollutants:
    valid_data = df[["station", pollutant]].dropna()

    tukey = pairwise_tukeyhsd(
        endog=valid_data[pollutant],
        groups=valid_data["station"],
        alpha=0.05,
    )
    tukey_table = pd.DataFrame(
        tukey.summary().data[1:],
        columns=tukey.summary().data[0],
    )
    tukey_table = tukey_table[tukey_table["reject"] == True].copy()
    tukey_table.insert(0, "pollutant", pollutant)
    significant_pairwise_results.append(tukey_table)

    station_values = [
        group[pollutant].dropna().to_numpy()
        for _, group in df.groupby("station", sort=True)
    ]
    levene_statistic, levene_p_value = levene(*station_values, center="median")

    residuals = valid_data[pollutant] - valid_data.groupby("station")[pollutant].transform("mean")
    residual_sample = residuals.sample(
        n=min(5000, len(residuals)), random_state=42
    )
    normality_statistic, normality_p_value = normaltest(residual_sample)

    assumption_results.append({
        "pollutant": pollutant,
        "Levene statistic": levene_statistic,
        "Levene p-value": levene_p_value,
        "Residual normality statistic": normality_statistic,
        "Residual normality p-value": normality_p_value,
        "equal variances at alpha=0.05": levene_p_value >= 0.05,
        "normal residuals at alpha=0.05": normality_p_value >= 0.05,
    })

significant_pairwise_results = pd.concat(
    significant_pairwise_results, ignore_index=True
)
assumption_results = pd.DataFrame(assumption_results)

pairwise_counts = (
    significant_pairwise_results.groupby("pollutant")
    .size()
    .reset_index(name="significant pair count")
)

top_pairwise_differences = (
    significant_pairwise_results.assign(
        absolute_meandiff=lambda table: table["meandiff"].abs()
    )
    .sort_values(["pollutant", "absolute_meandiff"], ascending=[True, False])
    .groupby("pollutant", group_keys=False)
    .head(3)
    [["pollutant", "group1", "group2", "meandiff", "p-adj", "lower", "upper"]]
)

print("Number of significant Tukey HSD pairs by pollutant")
display(pairwise_counts)

print("Three largest significant mean differences per pollutant")
display(top_pairwise_differences)

print("Assumption test results")
display(assumption_results.round(6))

## 9. Post-Hoc Interpretation

Use the current Tukey HSD results above to identify which station pairs differ in their daily pollutant summaries. The results depend on the current `aqi_daily` records; rerun the notebook before drawing conclusions from them.
